In [0]:
%sql
CREATE OR REPLACE TABLE parking_sessions_source (
session_id INT,
zone_id STRING,
vehicle_type STRING,
duration_hours INT,
parking_fee INT,
status STRING
)
USING DELTA;
INSERT INTO parking_sessions_source VALUES
(101, 'Z01', 'Car', 3, 150, 'Completed'),
(102, 'Z02', 'Bike', 2, 40, 'Completed'),
(103, 'Z03', 'Car', 5, 250, 'Completed'),
(104, 'Z01', 'SUV', 4, 240, 'Active'),
(105, 'Z04', 'Bike', 1, 20, 'Completed'),
(106, 'Z02', 'Car', 6, 300, 'Completed'),
(107, 'Z03', 'SUV', 2, 120, 'Cancelled'),
(108, 'Z05', 'Car', 3, 150, 'Completed'),
(109, 'Z01', 'Bike', 4, 80, 'Completed'),
(110, 'Z04', 'SUV', 5, 300, 'Active'),
(111, 'Z02', 'Car', 2, 100, 'Completed'),

(112, 'Z05', 'Bike', 3, 60, 'Cancelled'),
(113, 'Z03', 'Car', 7, 350, 'Completed'),
(114, 'Z01', 'SUV', 3, 180, 'Completed'),
(115, 'Z04', 'Bike', 2, 40, 'Active'),
(116, 'Z05', 'Car', 4, 200, 'Completed'),
(117, 'Z02', 'SUV', 6, 360, 'Completed'),
(118, 'Z03', 'Bike', 1, 20, 'Completed'),
(119, 'Z04', 'Car', 5, 250, 'Completed'),
(120, 'Z05', 'SUV', 2, 120, 'Active');

num_affected_rows,num_inserted_rows
20,20


In [0]:
%sql
CREATE OR REPLACE TABLE parking_sessions_work
USING DELTA
AS
SELECT * FROM parking_sessions_source;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE parking_zones (
zone_id STRING,
zone_name STRING,
city STRING,
capacity INT
)
USING DELTA;
INSERT INTO parking_zones VALUES
('Z01', 'Central Plaza', 'Hyderabad', 100),
('Z02', 'Metro Parking', 'Bengaluru', 150),
('Z03', 'City Mall', 'Mumbai', 200),
('Z04', 'Airport Parking', 'Delhi', 250),
('Z05', 'Tech Park', 'Pune', 120);

num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_parking_sessions AS
SELECT *
FROM VALUES
(102, 'Z02', 'Bike', 3, 60, 'Completed'),
(104, 'Z01', 'SUV', 5, 300, 'Completed'),
(121, 'Z03', 'Car', 4, 200, 'Completed'),
(122, 'Z05', 'Bike', 2, 40, 'Active')
AS t(
session_id,
zone_id,
vehicle_type,
duration_hours,
parking_fee,
status
);

Section A — SQL and DataFrames

In [0]:
%sql
select * from parking_sessions_source;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
104,Z01,SUV,4,240,Active
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
107,Z03,SUV,2,120,Cancelled
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
110,Z04,SUV,5,300,Active


In [0]:
%sql
select session_id,vehicle_type,parking_fee from parking_sessions_source;

session_id,vehicle_type,parking_fee
101,Car,150
102,Bike,40
103,Car,250
104,SUV,240
105,Bike,20
106,Car,300
107,SUV,120
108,Car,150
109,Bike,80
110,SUV,300


In [0]:
%sql
select * from parking_sessions_source where parking_fee>100 and status="Completed";

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
103,Z03,Car,5,250,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
113,Z03,Car,7,350,Completed
114,Z01,SUV,3,180,Completed
116,Z05,Car,4,200,Completed
117,Z02,SUV,6,360,Completed
119,Z04,Car,5,250,Completed


In [0]:
%sql
select vehicle_type,sum(parking_fee) as total_parking_fees,count(*) as no_of_completed_sessons from parking_sessions_source where status="Completed" group by vehicle_type;

vehicle_type,total_parking_fees,no_of_completed_sessons
Car,1750,8
Bike,160,4
SUV,540,2


In [0]:
from pyspark.sql.functions import when,col

In [0]:
parking_source_df=spark.table("parking_sessions_source")
parking_source_df=parking_source_df.withColumn("fee_category",when(col("parking_fee")>=200,"High").otherwise("Low"))
parking_source_df.show()

+----------+-------+------------+--------------+-----------+---------+------------+
|session_id|zone_id|vehicle_type|duration_hours|parking_fee|   status|fee_category|
+----------+-------+------------+--------------+-----------+---------+------------+
|       101|    Z01|         Car|             3|        150|Completed|         Low|
|       102|    Z02|        Bike|             2|         40|Completed|         Low|
|       103|    Z03|         Car|             5|        250|Completed|        High|
|       104|    Z01|         SUV|             4|        240|   Active|        High|
|       105|    Z04|        Bike|             1|         20|Completed|         Low|
|       106|    Z02|         Car|             6|        300|Completed|        High|
|       107|    Z03|         SUV|             2|        120|Cancelled|         Low|
|       108|    Z05|         Car|             3|        150|Completed|         Low|
|       109|    Z01|        Bike|             4|         80|Completed|      

Section B — Delta Tables and CRUD

In [0]:
%sql
create or replace table parking_summary using delta
as select * from parking_sessions_work where status="Completed";

num_affected_rows,num_inserted_rows


In [0]:
%sql
insert into parking_sessions_work values(
    123,
    "Z01",
    "Car",
    3,
    150,
    "Active"
)

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
update parking_sessions_work set status="Completed",parking_fee=280 where session_id=104;

num_affected_rows
1


In [0]:
%sql
delete from parking_sessions_work where status="Cancelled";
select * from parking_sessions_work;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
110,Z04,SUV,5,300,Active
111,Z02,Car,2,100,Completed
113,Z03,Car,7,350,Completed


Section C — MERGE and Delta Operations

In [0]:
%sql
select * from incoming_parking_sessions;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,3,60,Completed
104,Z01,SUV,5,300,Completed
121,Z03,Car,4,200,Completed
122,Z05,Bike,2,40,Active


In [0]:
%sql
select i.* from incoming_parking_sessions i inner join parking_sessions_work w on i.session_id=w.session_id;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,3,60,Completed
104,Z01,SUV,5,300,Completed


In [0]:
%sql
merge into parking_sessions_work as target 
using incoming_parking_sessions as source 
on target.session_id=source.session_id
when matched then update set
    target.zone_id = source.zone_id,
    target.vehicle_type = source.vehicle_type,
    target.duration_hours = source.duration_hours,
    target.parking_fee = source.parking_fee,
    target.status = source.status
when not matched then insert (
    session_id,
    zone_id,
    vehicle_type,
    duration_hours,
    parking_fee,
    status
)
VALUES (
    source.session_id,
    source.zone_id,
    source.vehicle_type,
    source.duration_hours,
    source.parking_fee,
    source.status
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,2,0,2


In [0]:
%sql
describe history parking_sessions_work;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-11-06T06:04:20.000Z,72288837415030,sivarubiniuthamaraj@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4219535361106174),fc55f391-e8a9-48b9-983e-5331d4ecae06,1106-060041-h3jhvim7-v2n,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 9875, p25FileSize -> 2287, numDeletionVectorsRemoved -> 1, minFileSize -> 2287, numAddedFiles -> 1, maxFileSize -> 2287, p75FileSize -> 2287, p50FileSize -> 2287, numAddedBytes -> 2287)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
5,2026-11-06T06:04:17.000Z,72288837415030,sivarubiniuthamaraj@gmail.com,MERGE,"Map(predicate -> [""(session_id#10210 = session_id#10192)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(4219535361106174),fc55f391-e8a9-48b9-983e-5331d4ecae06,1106-060041-h3jhvim7-v2n,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 7597, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 10557, materializeSourceTimeMs -> 479, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 6429, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3490)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
4,2026-11-06T05:12:37.000Z,72288837415030,sivarubiniuthamaraj@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4219535361106174),6b67643c-9bb3-4f53-9634-af9fe458c6cf,1106-043919-3fiacwz3-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 6092, p25FileSize -> 2278, numDeletionVectorsRemoved -> 1, minFileSize -> 2278, numAddedFiles -> 1, maxFileSize -> 2278, p75FileSize -> 2278, p50FileSize -> 2278, numAddedBytes -> 2278)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
3,2026-11-06T05:12:35.000Z,72288837415030,sivarubiniuthamaraj@gmail.com,DELETE,"Map(predicate -> [""(status#13680 = Cancelled)""])",null,List(4219535361106174),6b67643c-9bb3-4f53-9634-af9fe458c6cf,1106-043919-3fiacwz3-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1139, numDeletionVectorsUpdated -> 1, numDeletedRows -> 2, scanTimeMs -> 710, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 429)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
2,2026-11-06T05:09:22.000Z,72288837415030,sivarubiniuthamaraj@gmail.com,UPDATE,"Map(predicate -> [""(session_id#13180 = 104)""])",null,List(4219535361106174),8ff29b8e-add9-435f-aa66-8739f5c6152c,1106-043919-3fiacwz3-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 4155, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1905, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1932, rewriteTimeMs -> 2220)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-06T05:07:37.000Z,72288837415030,sivarubiniuthamaraj@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(4219535361106174),2c8d5d1a-7623-405b-9405-0555bd20e9a5,1106-043919-3fiacwz3-v2n,0,WriteSerializable,true

In [0]:
%sql
select * from parking_sessions_work version as of 4 where session_id=102;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,2,40,Completed


In [0]:
%sql
select * from parking_sessions_work where session_id=102;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,3,60,Completed


Section D — Tables, Views and Optimization

In [0]:
%sql
create or replace view completed_parking_view as select * from parking_sessions_work where status="Completed";

In [0]:
%sql
create or replace temp view zone_z01_view as 
select * from parking_sessions_work where zone_id="Z01";
select * from zone_z01_view;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
114,Z01,SUV,3,180,Completed
123,Z01,Car,3,150,Active
104,Z01,SUV,5,300,Completed


In [0]:
%sql
create or replace temp view high_fee_view as
select * from parking_sessions_work where parking_fee > 200;
select * from high_fee_view;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
103,Z03,Car,5,250,Completed
106,Z02,Car,6,300,Completed
110,Z04,SUV,5,300,Active
113,Z03,Car,7,350,Completed
117,Z02,SUV,6,360,Completed
119,Z04,Car,5,250,Completed
104,Z01,SUV,5,300,Completed


In [0]:
%sql
select p.zone_name,p.city,w.vehicle_type,w.parking_Fee from parking_sessions_work w inner join parking_zones p on w.zone_id=p.zone_id;

zone_name,city,vehicle_type,parking_Fee
Central Plaza,Hyderabad,Car,150
City Mall,Mumbai,Car,250
Airport Parking,Delhi,Bike,20
Metro Parking,Bengaluru,Car,300
Tech Park,Pune,Car,150
Central Plaza,Hyderabad,Bike,80
Airport Parking,Delhi,SUV,300
Metro Parking,Bengaluru,Car,100
City Mall,Mumbai,Car,350
Central Plaza,Hyderabad,SUV,180


In [0]:
%sql
describe detail parking_sessions_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,01361f9a-38f0-4be7-b684-efa589a8deb6,workspace.default.parking_sessions_work,null,,2026-11-06T04:40:59.108Z,2026-11-06T06:04:20.000Z,List(),List(),1,2287,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
optimize parking_sessions_work;

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1793947693355, 1793947694166, 8, 0, null, List(0, 0), null, 6, 6, 0, 0, null, null, 0)"


In [0]:
%sql
describe detail parking_sessions_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,01361f9a-38f0-4be7-b684-efa589a8deb6,workspace.default.parking_sessions_work,null,,2026-11-06T04:40:59.108Z,2026-11-06T06:04:20.000Z,List(),List(),1,2287,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


Section E — Structured Streaming and DLT

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.streaming_checkpoints")

rate_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

query = rate_df.writeStream \
    .format("console") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/workspace/default/streaming_checkpoints/rate_console") \
    .trigger(availableNow=True) \
    .start()

In [0]:
from pyspark.sql.functions import col

transformed_df = rate_df \
    .withColumn("amount", col("value") * 100) \
    .filter(col("amount") > 300)

query = transformed_df.writeStream \
    .format("console") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/workspace/default/streaming_checkpoints/rate_transformed_console") \
    .trigger(availableNow=True) \
    .start()

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE bronze_parking
AS
SELECT *
FROM STREAM(parking_sessions_source);

message
The operation was successfully executed.


In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE silver_parking
AS
SELECT *
FROM STREAM(bronze_parking)
WHERE status = 'Completed';

message
The operation was successfully executed.


In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW gold_parking
AS
SELECT
    zone_id,
    COUNT(*) AS total_completed_sessions,
    SUM(parking_fee) AS total_parking_revenue
FROM silver_parking
GROUP BY zone_id;

result
The operation was successfully executed.
